In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

from catboost import CatBoostRegressor

DATA_PATH = Path(
    "data/synthetic/"
    "gta_store_product_week_history_v1.parquet"
)

MODEL_DIR = Path("models")
RESULTS_DIR = Path("results")

MODEL_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

df = pd.read_parquet(DATA_PATH)

df["week_start"] = pd.to_datetime(
    df["week_start"]
)

print(df.shape)
print(df["week_start"].min())
print(df["week_start"].max())

display(df.head())

(468000, 54)
2024-09-16 00:00:00
2026-09-07 00:00:00


,store_id,store_name,city,population_5km,median_household_income_5km,offices,commercial,universities,colleges,population_5km_z,...,unit_margin,profit,expected_profit_control,expected_profit_treated,true_incremental_profit,true_incremental_units,store_context_origin,weather_origin,sales_outcome_origin,simulation_version
0,3106,Toronto Dufferin Mall,Toronto,593613.628819,86252.286112,940,172,19,19,2.291383,...,4.306873,103.364951,91.671057,91.101838,-0.569219,1.915635,real,real,synthetic,v1
1,3106,Toronto Dufferin Mall,Toronto,593613.628819,86252.286112,940,172,19,19,2.291383,...,3.926725,106.021570,91.671057,91.101838,-0.569219,1.915635,real,real,synthetic,v1
2,3106,Toronto Dufferin Mall,Toronto,593613.628819,86252.286112,940,172,19,19,2.291383,...,4.306873,116.285570,91.671057,91.101838,-0.569219,1.915635,real,real,synthetic,v1
3,3106,Toronto Dufferin Mall,Toronto,593613.628819,86252.286112,940,172,19,19,2.291383,...,4.306873,94.751205,91.671057,91.101838,-0.569219,1.915635,real,real,synthetic,v1
4,3106,Toronto Dufferin Mall,Toronto,593613.628819,86252.286112,940,172,19,19,2.291383,...,3.926725,90.314670,91.671057,91.101838,-0.569219,1.915635,real,real,synthetic,v1


# Sort properly

In [2]:
df = (
    df
    .sort_values(
        [
            "store_id",
            "product_id",
            "week_start"
        ]
    )
    .reset_index(drop=True)
)

# Create historical sales features

In [3]:
df["lag_1"] = (
    df.groupby(
        ["store_id", "product_id"]
    )["units_sold"]
    .shift(1)
)

In [4]:
df["lag_4"] = (
    df.groupby(
        ["store_id", "product_id"]
    )["units_sold"]
    .shift(4)
)

In [5]:
df["rolling_mean_4"] = (
    df.groupby(
        ["store_id", "product_id"]
    )["units_sold"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            4,
            min_periods=1
        )
        .mean()
    )
)

In [6]:
df["rolling_mean_8"] = (
    df.groupby(
        ["store_id", "product_id"]
    )["units_sold"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            8,
            min_periods=1
        )
        .mean()
    )
)

# Calendar features

In [7]:
df["month"] = df["week_start"].dt.month
df["week_of_year"] = (
    df["week_start"]
    .dt.isocalendar()
    .week
    .astype(int)
)

# Remove first observations without history

In [8]:
df = df[
    df["lag_1"].notna()
].copy()

print(df.shape)

(463500, 59)


# Time-based split

In [9]:
weeks = np.array(
    sorted(
        df["week_start"].unique()
    )
)

print("Weeks available:", len(weeks))

Weeks available: 103


In [10]:
val_start = weeks[-24]
test_start = weeks[-12]

train = df[
    df["week_start"] < val_start
].copy()

val = df[
    (df["week_start"] >= val_start)
    & (df["week_start"] < test_start)
].copy()

test = df[
    df["week_start"] >= test_start
].copy()

print("Train:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

print(
    "Train dates:",
    train["week_start"].min(),
    "→",
    train["week_start"].max()
)

print(
    "Validation:",
    val["week_start"].min(),
    "→",
    val["week_start"].max()
)

print(
    "Test:",
    test["week_start"].min(),
    "→",
    test["week_start"].max()
)

Train: (355500, 59)
Validation: (54000, 59)
Test: (54000, 59)
Train dates: 2024-09-23 00:00:00 → 2026-03-23 00:00:00
Validation: 2026-03-30 00:00:00 → 2026-06-15 00:00:00
Test: 2026-06-22 00:00:00 → 2026-09-07 00:00:00


# Baseline

In [11]:
def evaluate(y_true, y_pred):

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    wmape = (
        np.abs(
            y_true - y_pred
        ).sum()
        /
        np.abs(y_true).sum()
        * 100
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "WMAPE_pct": wmape
    }

In [12]:
baseline_metrics = evaluate(
    val["units_sold"],
    val["lag_1"]
)

baseline_metrics

{'MAE': 4.356740740740741,
 'RMSE': np.float64(5.677414643747366),
 'WMAPE_pct': np.float64(31.783713386733098)}

# Choose ML features

In [13]:
categorical_features = [
    "store_id",
    "product_id",
    "category",
    "city"
]

In [14]:
numeric_features = [
    # Price/promotion
    "regular_price",
    "selling_price",
    "discount_pct",
    "promotion",

    # Inventory
    "inventory",

    # Historical demand
    "lag_1",
    "lag_4",
    "rolling_mean_4",
    "rolling_mean_8",

    # Weather
    "mean_temperature",
    "total_rain",
    "total_snow",

    # Calendar
    "month",
    "week_of_year",

    # Real 5-km area
    "population_5km",
    "median_household_income_5km",
    "offices",
    "commercial",
    "universities",
    "colleges"
]

In [15]:
FEATURES = (
    categorical_features
    + numeric_features
)

TARGET = "units_sold"

# Prepare train/validation/test

In [16]:
X_train = train[FEATURES].copy()
y_train = train[TARGET].copy()

X_val = val[FEATURES].copy()
y_val = val[TARGET].copy()

X_test = test[FEATURES].copy()
y_test = test[TARGET].copy()

In [17]:
for col in categorical_features:

    X_train[col] = (
        X_train[col]
        .fillna("Unknown")
        .astype(str)
    )

    X_val[col] = (
        X_val[col]
        .fillna("Unknown")
        .astype(str)
    )

    X_test[col] = (
        X_test[col]
        .fillna("Unknown")
        .astype(str)
    )

# Train CatBoost

In [18]:
model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,

    loss_function="RMSE",
    eval_metric="RMSE",

    random_seed=42,

    verbose=100
)

In [19]:
model.fit(
    X_train,
    y_train,

    cat_features=categorical_features,

    eval_set=(
        X_val,
        y_val
    ),

    early_stopping_rounds=100,

    use_best_model=True
)

0:	learn: 6.8934268	test: 6.9331229	best: 6.9331229 (0)	total: 104ms	remaining: 1m 43s
100:	learn: 3.9303336	test: 3.9705953	best: 3.9705953 (100)	total: 4.05s	remaining: 36s
200:	learn: 3.8889049	test: 3.9364163	best: 3.9364163 (200)	total: 8.37s	remaining: 33.3s
300:	learn: 3.8585357	test: 3.9181134	best: 3.9181134 (300)	total: 12.9s	remaining: 29.9s
400:	learn: 3.8348607	test: 3.9093417	best: 3.9093417 (400)	total: 17.5s	remaining: 26.1s
500:	learn: 3.8189561	test: 3.9050031	best: 3.9050031 (500)	total: 22.2s	remaining: 22.1s
600:	learn: 3.8065148	test: 3.9024586	best: 3.9024586 (600)	total: 27s	remaining: 17.9s
700:	learn: 3.7945304	test: 3.9010794	best: 3.9009971 (697)	total: 31.9s	remaining: 13.6s
800:	learn: 3.7837319	test: 3.9004254	best: 3.9004141 (796)	total: 36.8s	remaining: 9.14s
900:	learn: 3.7741734	test: 3.9003390	best: 3.9002027 (856)	total: 41.8s	remaining: 4.6s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 3.900202687
bestIteration = 856

Shrink m

CatBoostRegressor(depth=8, eval_metric='RMSE', iterations=1000, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

# Validation predictions

In [20]:
val_pred = model.predict(
    X_val
)

val_pred = np.clip(
    val_pred,
    0,
    None
)

In [21]:
catboost_val_metrics = evaluate(
    y_val,
    val_pred
)

catboost_val_metrics

{'MAE': 3.0624839393101686,
 'RMSE': np.float64(3.9002026869817725),
 'WMAPE_pct': np.float64(22.34172689420998)}

# Compare baseline vs ML

In [22]:
comparison = pd.DataFrame([
    {
        "model": "Lag-1 Baseline",
        **baseline_metrics
    },

    {
        "model": "CatBoost",
        **catboost_val_metrics
    }
])

display(comparison)

,model,MAE,RMSE,WMAPE_pct
0,Lag-1 Baseline,4.356741,5.677415,31.783713
1,CatBoost,3.062484,3.900203,22.341727


# Final test

In [23]:
test_pred = model.predict(
    X_test
)

test_pred = np.clip(
    test_pred,
    0,
    None
)

test_metrics = evaluate(
    y_test,
    test_pred
)

test_metrics

{'MAE': 3.0900259083861585,
 'RMSE': np.float64(3.94393328851571),
 'WMAPE_pct': np.float64(21.900206853614314)}

# Feature importance

In [24]:
importance = pd.DataFrame({
    "feature": FEATURES,
    "importance":
        model.get_feature_importance()
})

importance = (
    importance
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    importance.head(20)
)

,feature,importance
0,rolling_mean_8,33.640038
1,inventory,30.801291
2,product_id,6.952108
3,promotion,5.917642
4,discount_pct,4.112112
5,rolling_mean_4,4.047669
6,category,2.278150
7,store_id,1.886222
8,mean_temperature,1.547518
9,lag_1,1.308004


# Save model

In [25]:
MODEL_PATH = (
    MODEL_DIR /
    "catboost_gta_demand_v1.cbm"
)

model.save_model(
    MODEL_PATH
)

print("Saved:", MODEL_PATH)

Saved: models/catboost_gta_demand_v1.cbm


# Save results

In [26]:
final_results = pd.DataFrame([
    {
        "split": "validation",
        "model": "Lag-1 Baseline",
        **baseline_metrics
    },

    {
        "split": "validation",
        "model": "CatBoost",
        **catboost_val_metrics
    },

    {
        "split": "test",
        "model": "CatBoost",
        **test_metrics
    }
])

RESULT_PATH = (
    RESULTS_DIR /
    "semisynthetic_demand_model_results_v1.csv"
)

final_results.to_csv(
    RESULT_PATH,
    index=False
)

display(final_results)

print("Saved:", RESULT_PATH)

,split,model,MAE,RMSE,WMAPE_pct
0,validation,Lag-1 Baseline,4.356741,5.677415,31.783713
1,validation,CatBoost,3.062484,3.900203,22.341727
2,test,CatBoost,3.090026,3.943933,21.900207


Saved: results/semisynthetic_demand_model_results_v1.csv


# Save test predictions

In [27]:
prediction_output = test[
    [
        "week_start",
        "store_id",
        "product_id",
        "category",
        "promotion",
        "units_sold"
    ]
].copy()

prediction_output["predicted_units"] = (
    test_pred
)

prediction_output[
    "absolute_error"
] = np.abs(
    prediction_output["units_sold"]
    - prediction_output["predicted_units"]
)

prediction_output.to_parquet(
    RESULTS_DIR /
    "demand_test_predictions_v1.parquet",
    index=False
)

display(
    prediction_output.head(20)
)

,week_start,store_id,product_id,category,promotion,units_sold,predicted_units,absolute_error
92,2026-06-22,1004,SYN_0001,Hot Beverages,1,23,19.912028,3.087972
93,2026-06-29,1004,SYN_0001,Hot Beverages,0,17,19.396265,2.396265
94,2026-07-06,1004,SYN_0001,Hot Beverages,0,25,18.726395,6.273605
95,2026-07-13,1004,SYN_0001,Hot Beverages,0,22,18.468685,3.531315
96,2026-07-20,1004,SYN_0001,Hot Beverages,1,27,21.295082,5.704918
97,2026-07-27,1004,SYN_0001,Hot Beverages,1,28,21.787220,6.212780
98,2026-08-03,1004,SYN_0001,Hot Beverages,0,22,19.505236,2.494764
99,2026-08-10,1004,SYN_0001,Hot Beverages,0,24,19.702631,4.297369
100,2026-08-17,1004,SYN_0001,Hot Beverages,0,17,18.332280,1.332280
101,2026-08-24,1004,SYN_0001,Hot Beverages,0,24,19.304973,4.695027
